# 10年定着予測 - 特徴量選択パイプライン（相関/VIF・Adversarial Validation/PSI・Boruta）

**背景**: 現在の最良は`28_relocation_mismatch_extended_extraction`のCatBoost + ブロックL_v2
（Public 0.529454）。特徴量数は441列に達しており、月次集約特徴量（16指標×mean/early/mid/late/
slope/diff/ratio等）には強く相関する列が多数含まれる、Train(2011-2014入社)/Test(2014-2017入社)の
分布シフトに脆弱な列が混在している可能性がある、重要度の低いノイズ列が探索空間を無駄に広げている
可能性がある、といった懸念がある。新規特徴量の発見が難しくなってきたフェーズを踏まえ、
**既存441列の質を上げる（削る）方向**を検証する。

## 3ステップの特徴量選択

1. **Step1: 相関/VIFフィルタ** — 数値特徴量ペアで|r|>0.95のものを検出し、目的変数との相関が
   低い方を削除する。
2. **Step2: Adversarial Validation / PSI** — Train/Testを分類するCatBoostモデルを学習し、
   重要度が高い特徴量ほど「Train/Testで分布が違う」＝ドリフトしていると判断する。PSI
   （Population Stability Index）でも同様に分布変化を定量化する。ただし、ドリフトしていても
   目的変数との相関が高い特徴量（初任給等、EDA v2で既知）は誤って削除しないよう、
   目的変数相関でフィルタする。
3. **Step3: 軽量Boruta** — 各特徴量をシャッフルしたシャドウ特徴量を作り、実特徴量の重要度が
   シャドウ特徴量群の最大重要度を上回るかを複数回（反復回数は計算コストを踏まえ削減するが、
   比較可能な範囲で）繰り返し判定し、勝率が低い特徴量を削除候補とする。

## 検証方法

Step1〜3を`28_`のベースライン（18_ + TF-IDF A_v1 + D_expanded + ブロックL_v2、441列、
80/20splitの学習期間IDのみで計算しリークを防ぐ）に順番に適用し、各段階でCatBoost + Optuna
（探索範囲は`18_`〜`32_`と同一のn_trials=25）の検証Log Lossがbaseline（441列、削減なし）を
上回るかを80/20・75/25の2つの時系列splitで比較する。

## 実行環境
Google Colab（CPU、ハイメモリ推奨）を想定。


In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 24.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 19.3 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "33_feature_selection_pipeline"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-11 09:29:55] [INFO] === [33_feature_selection_pipeline] 実験開始 ===


INFO:33_feature_selection_pipeline:=== [33_feature_selection_pipeline] 実験開始 ===


[2026-08-11 09:29:55] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


INFO:33_feature_selection_pipeline:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


[2026-08-11 09:29:55] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/33_feature_selection_pipeline_checkpoint.csv


INFO:33_feature_selection_pipeline:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/33_feature_selection_pipeline_checkpoint.csv


[2026-08-11 09:29:56] [INFO] チェックポイントは未作成（新規実行）


INFO:33_feature_selection_pipeline:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-11 09:30:01] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:33_feature_selection_pipeline:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-11 09:30:01] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:33_feature_selection_pipeline:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-11 09:30:01] [INFO] 定着率: 0.5647


INFO:33_feature_selection_pipeline:定着率: 0.5647


[2026-08-11 09:30:01] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:33_feature_selection_pipeline:Train IDs: 2761, Test IDs: 2502


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [7]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [8]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-11 09:30:01] [INFO] ------------------------------------------------------------


INFO:33_feature_selection_pipeline:------------------------------------------------------------


[2026-08-11 09:30:01] [INFO] split非依存の基本特徴量を生成中...


INFO:33_feature_selection_pipeline:split非依存の基本特徴量を生成中...


[2026-08-11 09:30:01] [INFO] ------------------------------------------------------------


INFO:33_feature_selection_pipeline:------------------------------------------------------------


[2026-08-11 09:37:40] [INFO] split非依存の基本特徴量生成完了


INFO:33_feature_selection_pipeline:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [9]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-11 09:37:40] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:33_feature_selection_pipeline:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-11 09:37:42] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:33_feature_selection_pipeline:入社時メモ: SVD累積寄与率=0.760


[2026-08-11 09:37:48] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:33_feature_selection_pipeline:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-11 09:37:50] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:33_feature_selection_pipeline:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-11 09:37:50] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:33_feature_selection_pipeline:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [10]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-11 09:37:50] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:33_feature_selection_pipeline:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-11 09:40:43] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:33_feature_selection_pipeline:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [11]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-11 09:40:44] [INFO] Persona単位の基本特徴量を生成中...


INFO:33_feature_selection_pipeline:Persona単位の基本特徴量を生成中...


[2026-08-11 09:40:44] [INFO] Persona単位の基本特徴量処理完了


INFO:33_feature_selection_pipeline:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [12]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-11 09:40:44] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:33_feature_selection_pipeline:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-11 09:40:44] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:33_feature_selection_pipeline:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-11 09:40:44] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:33_feature_selection_pipeline:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [13]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None):
    '''指定した分割比率で特徴量を組み立てる。extra_blocks: {"L1","L2"}のサブセット（通常はどちらか一方）'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")

✅ 部署Target Encoding・prepare_split関数定義完了


## 7. チェックポイント機能（`18_`〜`27_`と同一）

In [14]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=["config", "n_features", "val_score", "submission_path"])

def save_checkpoint_row(result):
    df = pd.DataFrame([result])
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']:.6f}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")

✅ チェックポイント関数定義完了


## 8. モデル実行関数（CatBoost、継続検証中のモデル）

`18_`のステップBでCatBoostが大差で最良だったため、本ノートブックではCatBoostのみで検証する。

In [15]:
def run_model_config(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=25, feature_subset=None):
    all_feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    feature_cols = all_feature_cols if feature_subset is None else [c for c in feature_subset if c in all_feature_cols]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]

    X_tr = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]
    X_test = test_features[feature_cols].fillna(-999)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    final_model = cb.CatBoostClassifier(
        **best_params, iterations=3000, random_seed=SEED, verbose=False,
        cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
    )
    final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    val_preds = final_model.predict_proba(X_va)[:, 1]
    test_preds = final_model.predict_proba(X_test)[:, 1]

    val_score = log_loss(y_va, val_preds)
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)

    val_pred_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy"
    np.save(val_pred_path, val_preds)

    logger.info(f"[{config_label}] n_features={len(feature_cols)}, val_score={val_score:.6f}")
    return {
        "config": config_label, "n_features": len(feature_cols), "val_score": val_score,
        "submission_path": str(sub_path), "val_pred_path": str(val_pred_path),
    }

print("✅ run_model_config関数定義完了")

✅ run_model_config関数定義完了


## 9. baseline特徴量の準備（`28_`と同一、441列）

特徴量選択はメインの提出split（80/20）の学習期間のデータのみを使って行う
（リーク防止。部署Target EncodingやブロックFと同じ方針）。

In [16]:
BASE_RATIO = 0.8
ag_train_base, ag_tuning_base, ttf_base = prepare_split(BASE_RATIO, extra_blocks={"L2"})
feature_cols_all = [c for c in ag_train_base.columns if c not in ["入社日", TARGET_COL]]
obj_cols_all = [c for c in feature_cols_all if ag_train_base[c].dtype == "object"]
num_cols_all = [c for c in feature_cols_all if c not in obj_cols_all]
logger.info(f"baseline特徴量数: {len(feature_cols_all)} (数値{len(num_cols_all)} / カテゴリ{len(obj_cols_all)})")
print(f"baseline特徴量数: {len(feature_cols_all)} (数値{len(num_cols_all)} / カテゴリ{len(obj_cols_all)})")

[2026-08-11 09:40:45] [INFO] baseline特徴量数: 441 (数値433 / カテゴリ8)


INFO:33_feature_selection_pipeline:baseline特徴量数: 441 (数値433 / カテゴリ8)


baseline特徴量数: 441 (数値433 / カテゴリ8)


## 10. Step1: 相関/VIFフィルタ

数値特徴量ペアで|r|>0.95のものを検出し、目的変数との相関(絶対値)が低い方を削除する
（ともに学習期間データのみで計算）。

In [17]:
def correlation_filter(df, num_cols, target, corr_threshold=0.95):
    X = df[num_cols]
    target_corr = X.corrwith(target).abs()
    corr_matrix = X.corr().abs()

    pairs = []
    for i in range(len(num_cols)):
        for j in range(i + 1, len(num_cols)):
            c1, c2 = num_cols[i], num_cols[j]
            r = corr_matrix.loc[c1, c2]
            if pd.notna(r) and r > corr_threshold:
                pairs.append((r, c1, c2))
    pairs.sort(key=lambda x: x[0], reverse=True)

    to_drop = set()
    for r, c1, c2 in pairs:
        if c1 in to_drop or c2 in to_drop:
            continue
        if target_corr.get(c1, 0) >= target_corr.get(c2, 0):
            to_drop.add(c2)
        else:
            to_drop.add(c1)
    return to_drop, pairs, target_corr


step1_drop, corr_pairs, target_corr_all = correlation_filter(
    ag_train_base, num_cols_all, ag_train_base[TARGET_COL], corr_threshold=0.95
)
logger.info(f"Step1: |r|>0.95のペア数={len(corr_pairs)}, 削除対象列数={len(step1_drop)}")
print(f"Step1: |r|>0.95のペア数={len(corr_pairs)}, 削除対象列数={len(step1_drop)}")
print("削除される列の例(上位10件、相関の強かった順):")
for r, c1, c2 in corr_pairs[:10]:
    dropped = c2 if c2 in step1_drop else c1
    kept = c1 if c2 in step1_drop else c2
    print(f"  r={r:.3f}: {dropped} を削除（{kept} を保持）")

feature_cols_step1 = [c for c in feature_cols_all if c not in step1_drop]
num_cols_step1 = [c for c in num_cols_all if c not in step1_drop]
logger.info(f"Step1後の特徴量数: {len(feature_cols_step1)}")
print(f"\nStep1後の特徴量数: {len(feature_cols_step1)} (441→{len(feature_cols_step1)})")

[2026-08-11 09:40:47] [INFO] Step1: |r|>0.95のペア数=229, 削除対象列数=85


INFO:33_feature_selection_pipeline:Step1: |r|>0.95のペア数=229, 削除対象列数=85


Step1: |r|>0.95のペア数=229, 削除対象列数=85
削除される列の例(上位10件、相関の強かった順):
  r=1.000: 有給取得日数_mid_mean を削除（有給取得日数_q2_mean_exp を保持）
  r=1.000: 有給取得率 を削除（有給取得日数_mean を保持）
  r=1.000: 有給取得日数_late_minus_early を削除（有給取得日数_late_mean を保持）
  r=1.000: 360度評価_親和度_q1_mean_exp を削除（360度評価_親和度_early_mean を保持）
  r=1.000: 360度評価_信頼度_q1_mean_exp を削除（360度評価_信頼度_early_mean を保持）
  r=1.000: 360度評価_主体度_q1_mean_exp を削除（360度評価_主体度_early_mean を保持）
  r=1.000: 360度評価_学習度_q1_mean_exp を削除（360度評価_学習度_early_mean を保持）
  r=1.000: 360度評価_共有貢献度_q1_mean_exp を削除（360度評価_共有貢献度_early_mean を保持）
  r=1.000: 360度評価者数_q1_mean_exp を削除（360度評価者数_early_mean を保持）
  r=1.000: 360度評価_信頼度_missing_rate を削除（360度評価_親和度_missing_rate を保持）
[2026-08-11 09:40:47] [INFO] Step1後の特徴量数: 356


INFO:33_feature_selection_pipeline:Step1後の特徴量数: 356



Step1後の特徴量数: 356 (441→356)


## 11. Step2: Adversarial Validation / PSI

Train(2011-2014入社)全2,761件とTest(2014-2017入社)全2,502件を分類するCatBoostモデルを
5-fold CVで学習し、重要度が高い特徴量ほどTrain/Testで分布が異なる（ドリフトしている）と判断する。
PSIでも同様に分布変化を定量化する。**目的変数との相関が高い特徴量（初任給等、分布シフトが
既知だが有用とEDA v2で確認済み）は誤って削除しないよう保護する。**

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


def adversarial_validation(train_df, test_df, feature_cols, obj_cols, seed=42, n_splits=5):
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    X_combined = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
    y_combined = np.array([0] * len(X_train) + [1] * len(X_test))

    num_cols_local = [c for c in feature_cols if c not in obj_cols]
    for c in num_cols_local:
        X_combined[c] = X_combined[c].fillna(-999)

    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(X_combined))
    importances = np.zeros(len(feature_cols))
    for tr_idx, va_idx in kf.split(X_combined, y_combined):
        m = cb.CatBoostClassifier(
            iterations=300, depth=6, learning_rate=0.1, random_seed=seed,
            verbose=False, cat_features=obj_cols, task_type="CPU",
        )
        m.fit(X_combined.iloc[tr_idx], y_combined[tr_idx])
        oof_preds[va_idx] = m.predict_proba(X_combined.iloc[va_idx])[:, 1]
        importances += np.array(m.get_feature_importance()) / n_splits

    auc = roc_auc_score(y_combined, oof_preds)
    imp_series = pd.Series(importances, index=feature_cols).sort_values(ascending=False)
    return auc, imp_series


# 特徴量選択はTrain全体(学習期間に限らない)とTest全体の分布差を見るため、split_ratio=1.0で
# 全train行を使う（ag_tuningは空になる）
ag_train_full, _, ttf_full = prepare_split(1.0, extra_blocks={"L2"})

obj_cols_step1 = [c for c in feature_cols_step1 if c in obj_cols_all]
adv_auc, adv_importance = adversarial_validation(ag_train_full, ttf_full, feature_cols_step1, obj_cols_step1, seed=SEED)
logger.info(f"Adversarial Validation AUC: {adv_auc:.4f}（0.5に近いほどTrain/Testの分布が近い）")
print(f"Adversarial Validation AUC: {adv_auc:.4f}")
print("\nTrain/Testを分類する上で重要度が高い特徴量(上位15件、ドリフト候補):")
print(adv_importance.head(15))

[2026-08-11 09:41:23] [INFO] Adversarial Validation AUC: 1.0000（0.5に近いほどTrain/Testの分布が近い）


INFO:33_feature_selection_pipeline:Adversarial Validation AUC: 1.0000（0.5に近いほどTrain/Testの分布が近い）


Adversarial Validation AUC: 1.0000

Train/Testを分類する上で重要度が高い特徴量(上位15件、ドリフト候補):
入社年                         91.168535
入社月                          3.994113
月例給与_円_volatility            0.146666
360度評価_親和度_missing_rate      0.112916
研修時間_std                     0.091687
同僚からのフィードバック_tfidf_svd_3     0.088652
研修時間_mean                    0.081898
同僚からのフィードバック_tfidf_svd_4     0.079386
顧客満足度評価_median               0.078635
情報共有件数_late_minus_early      0.075820
研修時間_kurtosis                0.072949
月例給与_円_late_early_ratio      0.071972
入社時メモ_tfidf_svd_4            0.057101
顧客満足度評価_cv                   0.056669
dept_target_enc              0.053013
dtype: float64


In [19]:
def calculate_psi(train_series, test_series, bins=10):
    if train_series.dtype == "object":
        train_counts = train_series.value_counts(normalize=True)
        test_counts = test_series.value_counts(normalize=True)
        all_cats = set(train_counts.index) | set(test_counts.index)
        psi = 0.0
        for cat in all_cats:
            p_train = max(train_counts.get(cat, 0), 1e-4)
            p_test = max(test_counts.get(cat, 0), 1e-4)
            psi += (p_test - p_train) * np.log(p_test / p_train)
        return psi

    train_valid = train_series.dropna()
    test_valid = test_series.dropna()
    if len(train_valid) < 10 or len(test_valid) < 10:
        return np.nan
    bin_edges = np.unique(np.percentile(train_valid, np.linspace(0, 100, bins + 1)))
    if len(bin_edges) < 3:
        return np.nan
    train_binned = pd.cut(train_valid, bins=bin_edges, include_lowest=True)
    test_binned = pd.cut(test_valid, bins=bin_edges, include_lowest=True)
    train_dist = train_binned.value_counts(normalize=True)
    test_dist = test_binned.value_counts(normalize=True)
    psi = 0.0
    for b in train_dist.index:
        p_train = max(train_dist.get(b, 0), 1e-4)
        p_test = max(test_dist.get(b, 0), 1e-4)
        psi += (p_test - p_train) * np.log(p_test / p_train)
    return psi


psi_results = {c: calculate_psi(ag_train_full[c], ttf_full[c]) for c in feature_cols_step1}
psi_series = pd.Series(psi_results).sort_values(ascending=False)
logger.info(f"PSI>0.25(顕著な変化)の特徴量数: {(psi_series > 0.25).sum()}")
print(f"PSI>0.25(顕著な変化)の特徴量数: {(psi_series > 0.25).sum()}")
print("\nPSI上位15件:")
print(psi_series.head(15))

# 削除候補: adversarial importance上位20% かつ PSI>0.25 の数値特徴量。
# ただし目的変数との相関が中央値以上(＝分布はシフトしていても予測に有用)の特徴量は保護する。
adv_top_n = max(1, int(len(feature_cols_step1) * 0.2))
high_adv_importance = set(adv_importance.head(adv_top_n).index)
high_psi = set(psi_series[psi_series > 0.25].index)
drop_candidates_step2 = (high_adv_importance & high_psi) & set(num_cols_step1)

target_corr_step1 = ag_train_full[num_cols_step1].corrwith(ag_train_full[TARGET_COL]).abs()
protected = set(target_corr_step1[target_corr_step1 >= target_corr_step1.median()].index)

step2_drop = drop_candidates_step2 - protected
step2_protected_but_flagged = drop_candidates_step2 & protected

logger.info(f"Step2削除候補: {len(drop_candidates_step2)}件、うち保護(目的変数相関が高い)された列: {len(step2_protected_but_flagged)}件")
logger.info(f"Step2で実際に削除: {len(step2_drop)}件")
print(f"\nStep2削除候補: {len(drop_candidates_step2)}件")
print(f"うち目的変数相関で保護された列: {sorted(step2_protected_but_flagged)}")
print(f"Step2で実際に削除する列: {sorted(step2_drop)}")

feature_cols_step2 = [c for c in feature_cols_step1 if c not in step2_drop]
print(f"\nStep2後の特徴量数: {len(feature_cols_step2)} ({len(feature_cols_step1)}→{len(feature_cols_step2)})")

[2026-08-11 09:41:25] [INFO] PSI>0.25(顕著な変化)の特徴量数: 10


INFO:33_feature_selection_pipeline:PSI>0.25(顕著な変化)の特徴量数: 10


PSI>0.25(顕著な変化)の特徴量数: 10

PSI上位15件:
入社年                              12.111723
cluster                           0.752683
360度評価者数_late_early_ratio         0.444969
360度評価_信頼度_early_mean             0.383299
360度評価_共有貢献度_early_mean           0.380392
360度評価_主体度_late_minus_early       0.352273
360度評価_学習度_late_early_ratio       0.336602
月例給与_等級内偏差                        0.306622
360度評価_信頼度_late_early_ratio       0.303003
360度評価_共有貢献度_late_early_ratio     0.268632
360度評価_主体度_early_mean             0.230891
初任給_区分内偏差                         0.220300
月例給与_円_q4_mean_exp                0.188973
dept_size                         0.184979
360度評価_信頼度_late_minus_early       0.164518
dtype: float64
[2026-08-11 09:41:25] [INFO] Step2削除候補: 3件、うち保護(目的変数相関が高い)された列: 2件


INFO:33_feature_selection_pipeline:Step2削除候補: 3件、うち保護(目的変数相関が高い)された列: 2件


[2026-08-11 09:41:25] [INFO] Step2で実際に削除: 1件


INFO:33_feature_selection_pipeline:Step2で実際に削除: 1件



Step2削除候補: 3件
うち目的変数相関で保護された列: ['cluster', '月例給与_等級内偏差']
Step2で実際に削除する列: ['入社年']

Step2後の特徴量数: 355 (356→355)


## 12. Step3: 軽量Boruta

各特徴量をシャッフルしたシャドウ特徴量を作り、実特徴量の重要度がシャドウ特徴量群の最大重要度を
上回るか（"勝ち"）を判定する。計算コストを踏まえ反復回数を12回に抑える（本来のBorutaは100回前後）。
12回中の勝率が低い特徴量を削除候補とする。

In [20]:
def boruta_lite(df, feature_cols, target_col, obj_cols, n_iterations=12, seed=42):
    real_cols = list(feature_cols)
    num_cols_local = [c for c in real_cols if c not in obj_cols]
    hits = pd.Series(0, index=real_cols)
    rng = np.random.RandomState(seed)
    target = df[target_col]

    base_real = df[real_cols].copy()
    for c in num_cols_local:
        base_real[c] = base_real[c].fillna(-999)

    for it in range(n_iterations):
        shadow = base_real.apply(lambda col: rng.permutation(col.values), axis=0)
        shadow.columns = [f"shadow_{c}" for c in real_cols]
        combined = pd.concat([base_real.reset_index(drop=True), shadow.reset_index(drop=True)], axis=1)
        shadow_obj_cols = [f"shadow_{c}" for c in obj_cols]
        for c in shadow_obj_cols:
            combined[c] = combined[c].astype(str)

        model = cb.CatBoostClassifier(
            iterations=300, depth=6, learning_rate=0.1, random_seed=seed + it,
            verbose=False, cat_features=obj_cols + shadow_obj_cols, task_type="CPU",
        )
        model.fit(combined, target.values)
        importances = pd.Series(model.get_feature_importance(), index=combined.columns)
        max_shadow_importance = importances[[f"shadow_{c}" for c in real_cols]].max()
        hits += (importances[real_cols] > max_shadow_importance).astype(int)
        logger.info(f"  Boruta iteration {it+1}/{n_iterations} 完了 (max_shadow_importance={max_shadow_importance:.4f})")

    return hits / n_iterations


logger.info("Step3: 軽量Boruta実行中(12反復)...")
obj_cols_step2 = [c for c in feature_cols_step2 if c in obj_cols_all]
hit_rate = boruta_lite(ag_train_base, feature_cols_step2, TARGET_COL, obj_cols_step2, n_iterations=12, seed=SEED)
hit_rate_sorted = hit_rate.sort_values()

print("勝率が低い特徴量(下位15件、削除候補):")
print(hit_rate_sorted.head(15))
print("\n勝率が高い特徴量(上位10件、重要と判定):")
print(hit_rate_sorted.sort_values(ascending=False).head(10))

# 反復数が少ないため厳密な統計検定は行わず、勝率<0.3を削除候補とする緩めの基準を採用
step3_drop = set(hit_rate[hit_rate < 0.3].index)
logger.info(f"Step3削除対象: {len(step3_drop)}件（勝率<0.3）")
print(f"\nStep3削除対象({len(step3_drop)}件、勝率<0.3): {sorted(step3_drop)[:30]}{'...' if len(step3_drop) > 30 else ''}")

feature_cols_step3 = [c for c in feature_cols_step2 if c not in step3_drop]
print(f"\nStep3後の特徴量数: {len(feature_cols_step3)} ({len(feature_cols_step2)}→{len(feature_cols_step3)})")
print(f"\n■ 全体まとめ: baseline {len(feature_cols_all)}列 → Step1後 {len(feature_cols_step1)}列 → Step2後 {len(feature_cols_step2)}列 → Step3後 {len(feature_cols_step3)}列")

[2026-08-11 09:41:26] [INFO] Step3: 軽量Boruta実行中(12反復)...


INFO:33_feature_selection_pipeline:Step3: 軽量Boruta実行中(12反復)...


[2026-08-11 09:41:38] [INFO]   Boruta iteration 1/12 完了 (max_shadow_importance=0.7740)


INFO:33_feature_selection_pipeline:  Boruta iteration 1/12 完了 (max_shadow_importance=0.7740)


[2026-08-11 09:41:51] [INFO]   Boruta iteration 2/12 完了 (max_shadow_importance=0.6101)


...

[2026-08-11 09:43:57] [INFO]   Boruta iteration 12/12 完了 (max_shadow_importance=0.6168)


INFO:33_feature_selection_pipeline:  Boruta iteration 12/12 完了 (max_shadow_importance=0.6168)


勝率が低い特徴量(下位15件、削除候補):
360度評価_親和度_q75         0.0
欠勤_連続フラグ               0.0
欠勤_最長連続月数              0.0
欠勤発生月数                 0.0
dept_size              0.0
dept_target_enc        0.0
cluster                0.0
360度評価_信頼度_iqr         0.0
360度評価_信頼度_q75         0.0
360度評価_信頼度_q25         0.0
360度評価_信頼度_kurtosis    0.0
初回評価月                  0.0
360度評価_親和度_q25         0.0
360度評価_親和度_kurtosis    0.0
360度評価_親和度_skew        0.0
dtype: float64

勝率が高い特徴量(上位10件、重要と判定):
残業時間_min             1.000000
転居x勤務地_状態_v2         1.000000
転居x勤務地_ダブル悪条件_v2     1.000000
残業時間_median          1.000000
初期職種                 1.000000
専攻分野                 1.000000
担当プロジェクト数_cv         1.000000
残業時間_cv              0.916667
残業時間_q3_mean_exp     0.833333
入社時メモ_tfidf_svd_1    0.833333
dtype: float64
[2026-08-11 09:43:57] [INFO] Step3削除対象: 340件（勝率<0.3）


INFO:33_feature_selection_pipeline:Step3削除対象: 340件（勝率<0.3）



Step3削除対象(340件、勝率<0.3): ['360度評価_主体度_acceleration_exp', '360度評価_主体度_cv', '360度評価_主体度_diff', '360度評価_主体度_early_mean', '360度評価_主体度_late_mean', '360度評価_主体度_late_minus_early', '360度評価_主体度_max', '360度評価_主体度_mean', '360度評価_主体度_median', '360度評価_主体度_min', '360度評価_主体度_q2_mean_exp', '360度評価_主体度_q3_mean_exp', '360度評価_主体度_q4_mean_exp', '360度評価_主体度_ratio', '360度評価_主体度_slope', '360度評価_主体度_std', '360度評価_信頼度_acceleration_exp', '360度評価_信頼度_cv', '360度評価_信頼度_diff', '360度評価_信頼度_early_mean', '360度評価_信頼度_iqr', '360度評価_信頼度_kurtosis', '360度評価_信頼度_late_early_ratio', '360度評価_信頼度_late_mean', '360度評価_信頼度_late_minus_early', '360度評価_信頼度_max', '360度評価_信頼度_mean', '360度評価_信頼度_median', '360度評価_信頼度_min', '360度評価_信頼度_q25']...

Step3後の特徴量数: 15 (355→15)

■ 全体まとめ: baseline 441列 → Step1後 356列 → Step2後 355列 → Step3後 15列


## 13. 各段階でのCatBoost検証（baseline / Step1後 / Step1+2後 / Step1+2+3後）× 2 split

各段階の特徴量リストを使い、`18_`〜`32_`と同一のCatBoost + Optuna(n_trials=25)で
80/20・75/25の2 splitを検証する。

In [21]:
SPLIT_RATIOS = {"split_80_20": 0.8, "split_75_25": 0.75}
FEATURE_STAGES = {
    "baseline_441": None,
    "after_step1": feature_cols_step1,
    "after_step1_2": feature_cols_step2,
    "after_step1_2_3": feature_cols_step3,
}

stage_results = []
for split_name, ratio in SPLIT_RATIOS.items():
    for stage_name, subset in FEATURE_STAGES.items():
        config_label = f"{split_name}_{stage_name}"
        def _run(ratio=ratio, subset=subset, config_label=config_label):
            logger.info(f"=== {config_label} ===")
            ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, extra_blocks={"L2"})
            return run_model_config(ag_train_data, ag_tuning_data, test_features_full, config_label, n_trials=25, feature_subset=subset)
        result = run_or_resume(config_label, _run)
        result["split"] = split_name
        result["stage"] = stage_name
        stage_results.append(result)

stage_df = pd.DataFrame(stage_results)
stage_pivot = stage_df.pivot(index="stage", columns="split", values="val_score")
stage_pivot["mean"] = stage_pivot[["split_80_20", "split_75_25"]].mean(axis=1)
stage_pivot["std"] = stage_pivot[["split_80_20", "split_75_25"]].std(axis=1)
stage_pivot = stage_pivot.reindex(["baseline_441", "after_step1", "after_step1_2", "after_step1_2_3"])
stage_pivot["mean_diff_vs_baseline"] = stage_pivot["mean"] - stage_pivot.loc["baseline_441", "mean"]
stage_pivot = stage_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("特徴量選択の段階別結果")
logger.info("=" * 60)
logger.info("\n" + stage_pivot.to_string())
print("\n■ 特徴量選択の段階別結果:")
print(stage_pivot.to_string())
print("\n※ baseline_441は28_のColab実測値(split_80_20=0.503065)と近い水準になるはず")
print("※ n_featuresは各resultのn_features列で確認できる")
print(stage_df[["stage", "split", "n_features", "val_score"]].to_string(index=False))

[2026-08-11 09:43:58] [INFO] === split_80_20_baseline_441 ===


INFO:33_feature_selection_pipeline:=== split_80_20_baseline_441 ===


[2026-08-11 09:47:54] [INFO] [split_80_20_baseline_441] n_features=441, val_score=0.503065


INFO:33_feature_selection_pipeline:[split_80_20_baseline_441] n_features=441, val_score=0.503065


[2026-08-11 09:47:54] [INFO] === split_80_20_after_step1 ===


...

[2026-08-11 10:04:51] [INFO] ============================================================


INFO:33_feature_selection_pipeline:============================================================


[2026-08-11 10:04:51] [INFO] 
split            split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
stage                                                                               
after_step1         0.509821     0.500894  0.505358  0.006313              -0.001238
baseline_441        0.510125     0.503065  0.506595  0.004992               0.000000
after_step1_2       0.515558     0.498087  0.506822  0.012354               0.000227
after_step1_2_3     0.516226     0.501788  0.509007  0.010209               0.002411


INFO:33_feature_selection_pipeline:
split            split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
stage                                                                               
after_step1         0.509821     0.500894  0.505358  0.006313              -0.001238
baseline_441        0.510125     0.503065  0.506595  0.004992               0.000000
after_step1_2       0.515558     0.498087  0.506822  0.012354               0.000227
after_step1_2_3     0.516226     0.501788  0.509007  0.010209               0.002411



■ 特徴量選択の段階別結果:
split            split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
stage                                                                               
after_step1         0.509821     0.500894  0.505358  0.006313              -0.001238
baseline_441        0.510125     0.503065  0.506595  0.004992               0.000000
after_step1_2       0.515558     0.498087  0.506822  0.012354               0.000227
after_step1_2_3     0.516226     0.501788  0.509007  0.010209               0.002411

※ baseline_441は28_のColab実測値(split_80_20=0.503065)と近い水準になるはず
※ n_featuresは各resultのn_features列で確認できる
          stage       split  n_features  val_score
   baseline_441 split_80_20         441   0.503065
    after_step1 split_80_20         356   0.500894
  after_step1_2 split_80_20         355   0.498087
after_step1_2_3 split_80_20          15   0.501788
   baseline_441 split_75_25         441   0.510125
    after_step1 split_75_25         356   0.509821
  after_step1_2 s

## 14. 総合結果・提出候補

split_80_20における各段階の提出ファイルを一覧化する。**baseline_441を明確に上回った最終段階
（できればStep3まで適用した段階、それが無理でも改善した最も進んだ段階）のみ提出候補とする**。
どの段階も改善しない場合は無理に提出せず、28_のL_v2構成をそのまま維持する。

In [22]:
split_80_20_rows = stage_df[stage_df["split"] == "split_80_20"].set_index("stage")
summary_rows = split_80_20_rows[["n_features", "val_score", "submission_path"]].reindex(
    ["baseline_441", "after_step1", "after_step1_2", "after_step1_2_3"]
).reset_index()

logger.info("=" * 60)
logger.info("総合結果（split_80_20、提出候補一覧）")
logger.info("=" * 60)
logger.info("\n" + summary_rows.to_string())
print("\n■ 総合結果（split_80_20、提出候補一覧）:")
print(summary_rows.to_string(index=False))
print(f"\n(参考) 28_ L_v2_extended(441列、削減なし): Public 0.529454（現時点の最良）")

summary_rows

[2026-08-11 10:04:51] [INFO] ============================================================


INFO:33_feature_selection_pipeline:============================================================


[2026-08-11 10:04:51] [INFO] 総合結果（split_80_20、提出候補一覧）


INFO:33_feature_selection_pipeline:総合結果（split_80_20、提出候補一覧）


[2026-08-11 10:04:51] [INFO] ============================================================


INFO:33_feature_selection_pipeline:============================================================


[2026-08-11 10:04:51] [INFO] 
             stage  n_features  val_score                                                                                                                 submission_path
0     baseline_441         441   0.503065     /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_baseline_441.csv
1      after_step1         356   0.500894      /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1.csv
2    after_step1_2         355   0.498087    /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1_2.csv
3  after_step1_2_3          15   0.501788  /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1_2_3.csv


INFO:33_feature_selection_pipeline:
             stage  n_features  val_score                                                                                                                 submission_path
0     baseline_441         441   0.503065     /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_baseline_441.csv
1      after_step1         356   0.500894      /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1.csv
2    after_step1_2         355   0.498087    /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1_2.csv
3  after_step1_2_3          15   0.501788  /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1_2_3.csv



■ 総合結果（split_80_20、提出候補一覧）:
          stage  n_features  val_score                                                                                                                submission_path
   baseline_441         441   0.503065    /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_baseline_441.csv
    after_step1         356   0.500894     /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1.csv
  after_step1_2         355   0.498087   /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1_2.csv
after_step1_2_3          15   0.501788 /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_33_feature_selection_pipeline_split_80_20_after_step1_2_3.csv

(参考) 28_ L_v2_extended(441列、削減なし): Public 0.529454（現時点の最良）


,stage,n_features,val_score,submission_path
0,baseline_441,441,0.503065,/content/drive/MyDrive/jaggle_2026/data/output...
1,after_step1,356,0.500894,/content/drive/MyDrive/jaggle_2026/data/output...
2,after_step1_2,355,0.498087,/content/drive/MyDrive/jaggle_2026/data/output...
3,after_step1_2_3,15,0.501788,/content/drive/MyDrive/jaggle_2026/data/output...


## 15. まとめ・次のアクション

1. 13節の段階別結果で、Step1（相関/VIF）・Step2（Adversarial/PSI）・Step3（Boruta）の
   どの段階まで適用するとbaseline_441を上回るか確認する。列数が減っても検証スコアが
   ほぼ同じ（ノイズ幅0.003〜0.006以内）であれば、それ自体もモデルの単純化として価値がある。
2. 最も改善した段階（列数が最小でスコアが最良の段階）の`split_80_20`提出ファイルをKaggleに
   提出し、Publicスコアを確認する。
3. どの段階もbaseline_441を明確に上回らない場合は無理に提出せず、
   **現時点の最良は引き続き28_ L_v2_extended（Public 0.529454）**とする。
4. 結果が出たら`data/output/submit_result_report.md`に追記し、`best_submission_status.md`も更新する。
5. Boruta（Step3）は反復回数を12回に抑えた簡易版のため、有望な結果が出た場合は反復回数を
   増やして（30〜50回程度）再検証する価値がある。

### バックログ（今回は着手しない）
- Step2のPSI閾値(0.25)・adversarial importance上位20%という基準の感度分析
- カテゴリ変数に対するStep2の適用（今回は数値変数のみに限定）
